# Machine Learning - A.A. 2025-2026
## Classificazione DPI - Rilevamento del Giubbotto di Sicurezza

Classificazione binaria (con/senza giubbotto) di immagini di persone. Etichette: `0 = without_vest`, `1 = with_vest`.

Pipeline: **Feature Extraction ResNet18** (lab *Feature Extraction*) + confronto dei classificatori dei Lab 3–7.

**Esperimenti:**
1. Modelli allenati e testati su **immagini sintetiche** (in-domain)
2. Stessi modelli testati su **foto reali** → *domain gap*
3. **Mixed training** (sintetico + reali di training) testato sulle reali di test

**Struttura dati:**
```
dataset/
  immagini sintetiche/{training, test}
  immagini reali/{training, test}
```

## 0. Setup di partenza


In [ ]:
import os
from os import path
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from torchvision import transforms, models
from torch.utils.data import DataLoader
from torch.utils.data.dataset import Dataset
from PIL import Image
from tqdm import tqdm

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.base import clone
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix, f1_score)

In [ ]:
BASE = Path("data") / "dataset"

SYN_TRAIN  = BASE / "immagini sintetiche" / "training"
SYN_TEST   = BASE / "immagini sintetiche" / "test"
REAL_TRAIN = BASE / "immagini reali" / "training"
REAL_TEST  = BASE / "immagini reali" / "test"

# classes[0] = "without_vest" (etichetta 0), classes[1] = "with_vest" (etichetta 1)
classi = ["without_vest", "with_vest"]

# estensioni immagini accettate
ESTENSIONI = {".jpg", ".jpeg", ".png", ".bmp"}

### Generazione dei file `labels.txt`

In [ ]:
# Legge il file di annotazioni 'nomefile,etichetta' presente nella cartella e restituisce la lista di coppie (nome_file, etichetta).
def leggi_annotazioni(cartella):
    cartella = Path(cartella)

    file_annotazioni = [f for f in cartella.glob("*.txt") if f.name != "labels.txt"]
    if not file_annotazioni:
        raise FileNotFoundError(f"Nessun file di annotazioni in {cartella}")

    righe = []
    for riga in open(file_annotazioni[0]):
        riga = riga.strip()
        if not riga:                      
            continue
        nome_file, etichetta = riga.split(",")
        righe.append((Path(nome_file.strip()).name, int(etichetta)))
    return righe


def scrivi_labels(percorso_file, righe):
    with open(percorso_file, "w") as f:
        for nome_file, etichetta in righe:
            f.write(f"{nome_file},{etichetta}\n")

# Per evitare di rigenerare le labels.txt ogni volta, si puo' impostare FORCE a True. In tal caso, anche se labels.txt esiste gia', verra' rigenerato.
FORCE = False

for cartella in [SYN_TRAIN, SYN_TEST, REAL_TRAIN, REAL_TEST]:
    file_labels = Path(cartella) / "labels.txt"

    if file_labels.exists() and not FORCE:
        numero_righe = sum(1 for _ in open(file_labels))
        continue

    righe = leggi_annotazioni(cartella)
    scrivi_labels(file_labels, righe)

## 1. Caricamento dati e pre-processing
Trasformazioni riprese dal lab *Feature Extraction* (224×224, normalizzazione ImageNet). La classe `ScenesDataset` è ripresa dal Lab 3, con una piccola aggiunta: se il file richiesto non esiste, prova le altre estensioni (`.jpg`/`.jpeg`).

In [ ]:
# Preprocess come ImageNet
trasformazioni = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],  # ImageNet stats
                        std=[0.229, 0.224, 0.225])
])

In [ ]:
# Carica le immagini elencate in un file 'nomefile,etichetta' e le restituisce come dizionario {"image": tensore, "label": etichetta}.
class ScenesDataset(Dataset):
    
    def __init__(self, cartella_immagini, file_labels, transform=None):
        self.cartella_immagini = cartella_immagini
        self.immagini = np.loadtxt(file_labels, dtype=str, delimiter=",")
        self.transform = transform

    def __getitem__(self, indice):
        nome_file, etichetta = self.immagini[indice]

        percorso_immagine = path.join(self.cartella_immagini, nome_file)

        if not os.path.exists(percorso_immagine):
            senza_estensione = os.path.splitext(percorso_immagine)[0]
            for estensione in (".jpg", ".jpeg", ".png", ".bmp"):
                if os.path.exists(senza_estensione + estensione):
                    percorso_immagine = senza_estensione + estensione
                    break

        immagine = Image.open(percorso_immagine).convert("RGB")
        if self.transform is not None:
            immagine = self.transform(immagine)

        return {"image": immagine, "label": int(etichetta)}

    def __len__(self):
        return len(self.immagini)

In [ ]:
# istanziamo i quattro dataset (uno per cartella)
dataset_syn_train  = ScenesDataset(SYN_TRAIN,  path.join(SYN_TRAIN,  "labels.txt"), transform=trasformazioni)
dataset_syn_test   = ScenesDataset(SYN_TEST,   path.join(SYN_TEST,   "labels.txt"), transform=trasformazioni)
dataset_real_train = ScenesDataset(REAL_TRAIN, path.join(REAL_TRAIN, "labels.txt"), transform=trasformazioni)
dataset_real_test  = ScenesDataset(REAL_TEST,  path.join(REAL_TEST,  "labels.txt"), transform=trasformazioni)

def crea_loader(dataset):
    return DataLoader(dataset, batch_size=64, shuffle=False)

loader_syn_train  = crea_loader(dataset_syn_train)
loader_syn_test   = crea_loader(dataset_syn_test)
loader_real_train = crea_loader(dataset_real_train)
loader_real_test  = crea_loader(dataset_real_test)

print("Sintetico  train:", len(dataset_syn_train),  " test:", len(dataset_syn_test))
print("Reale      train:", len(dataset_real_train), " test:", len(dataset_real_test))

## 2. Estrazione delle feature con ResNet18

Ci fermiamo ad `avgpool` (embedding 512-dim). Codice ripreso dal lab *Feature Extraction* (`In [5]`, `In [11]`); adattato l'accesso al batch (`batch["image"]`/`batch["label"]`) perché `ScenesDataset` restituisce un dizionario.


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
estrattore_feature = torch.nn.Sequential(*list(resnet.children())[:-1]).to(device).eval()  # [B,512,1,1]


def estrai_feature(loader, estrattore_feature, device):
    lista_X, lista_y = [], []
    for batch in tqdm(loader):
        immagini, etichette = batch["image"], batch["label"]
        with torch.no_grad():
            immagini = immagini.to(device)
            feature = estrattore_feature(immagini)   # [B, 512, 1, 1]
            feature = torch.flatten(feature, 1)      # [B, 512]
            lista_X.append(feature.cpu())
            lista_y.append(etichette)
    X = torch.cat(lista_X, dim=0).numpy()    # [N, 512]
    y = torch.cat(lista_y, dim=0).numpy()    # [N]
    return X, y

In [ ]:
FILE_FEATURE = "output/vest_feats.npz"
FORCE_EXTRACT = True   # True per riestrarre anche se il file esiste

In [ ]:
if os.path.exists(FILE_FEATURE) and not FORCE_EXTRACT:
    dati = np.load(FILE_FEATURE)
    X_syn_train, y_syn_train = dati["X_syn_train"], dati["y_syn_train"]
    X_syn_test,  y_syn_test  = dati["X_syn_test"],  dati["y_syn_test"]
    X_real_train, y_real_train = dati["X_real_train"], dati["y_real_train"]
    X_real_test,  y_real_test  = dati["X_real_test"],  dati["y_real_test"]
else:
    print("Sintetico train:"); X_syn_train, y_syn_train = estrai_feature(loader_syn_train,  estrattore_feature, device)
    print("Sintetico test:");  X_syn_test,  y_syn_test  = estrai_feature(loader_syn_test,   estrattore_feature, device)
    print("Reale train:");     X_real_train, y_real_train = estrai_feature(loader_real_train, estrattore_feature, device)
    print("Reale test:");      X_real_test,  y_real_test  = estrai_feature(loader_real_test,  estrattore_feature, device)
    os.makedirs("output", exist_ok=True)
    np.savez_compressed(FILE_FEATURE, X_syn_train=X_syn_train, y_syn_train=y_syn_train, X_syn_test=X_syn_test, y_syn_test=y_syn_test, X_real_train=X_real_train, y_real_train=y_real_train, X_real_test=X_real_test, y_real_test=y_real_test)

## 3. Logistic Regression sugli embedding (sintetico)

Addestriamo un classificatore lineare sugli embedding sintetici (cella `In [14]` del lab *Feature Extraction*, ripresa integralmente).


In [ ]:
clf = LogisticRegression(max_iter=5000)
clf.fit(X_syn_train, y_syn_train)
predizioni = clf.predict(X_syn_test)
print("Accuracy:", accuracy_score(y_syn_test, predizioni))
print(classification_report(y_syn_test, predizioni, target_names=classi))

## 4. Esperimento 1 — Confronto classificatori (sintetico, in-domain)

Firme riprese dai Lab 4/5/6/7/3. Normalizzazione con `StandardScaler` (Lab 6).

In [ ]:
# normalizzazione (fittata sul sintetico train)
scaler = StandardScaler()
X_syn_train_norm = scaler.fit_transform(X_syn_train)
X_syn_test_norm  = scaler.transform(X_syn_test)

classificatori = {
    "Logistic Regression": LogisticRegression(max_iter=5000),
    "Decision Tree (d=5)": DecisionTreeClassifier(criterion="gini", max_depth=5, random_state=42),
    "Random Forest":       RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42),
    "SVM Linear":          SVC(kernel="linear", C=0.1, random_state=42),
    "SVM RBF":             SVC(kernel="rbf", C=1.0, random_state=42),
    "AdaBoost":            AdaBoostClassifier(n_estimators=50, random_state=42),
    "MLP (512)":           MLPClassifier(hidden_layer_sizes=(512,), max_iter=500, random_state=42),
    "Deep MLP (512-512)":  MLPClassifier(hidden_layer_sizes=(512, 512), max_iter=500, random_state=42),
}

In [ ]:
print(f"{'Classificatore':<22} {'Train Acc':>10} {'Test Acc':>10} {'mF1':>8}")
print("-" * 54)
risultati = {}
for nome, modello in classificatori.items():
    modello.fit(X_syn_train_norm, y_syn_train)
    pred_train = modello.predict(X_syn_train_norm)
    pred_test  = modello.predict(X_syn_test_norm)
    acc_train = accuracy_score(y_syn_train, pred_train)
    acc_test  = accuracy_score(y_syn_test,  pred_test)
    mf1       = f1_score(y_syn_test, pred_test, average="macro")
    risultati[nome] = {"acc_train": acc_train, "acc_test": acc_test, "mf1": mf1, "pred": pred_test}
    print(f"{nome:<22} {acc_train:>10.4f} {acc_test:>10.4f} {mf1:>8.4f}")

In [ ]:
# confusion matrix di tutti i classificatori (sintetico test)
numero = len(risultati); colonne = 4; righe_grid = (numero + colonne - 1) // colonne
fig, assi = plt.subplots(righe_grid, colonne, figsize=(4*colonne, 3.5*righe_grid)); assi = assi.flatten()
for asse, (nome, res) in zip(assi, risultati.items()):
    cm = confusion_matrix(y_syn_test, res["pred"])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["no", "vest"], yticklabels=["no", "vest"], ax=asse, cbar=False)
    asse.set_title(f"{nome}\nacc={res['acc_test']:.3f}", fontsize=9)
    asse.set_xlabel("Pred", fontsize=8); asse.set_ylabel("Reale", fontsize=8)
for asse in assi[numero:]:
    asse.set_visible(False)
plt.suptitle("Esperimento 1 — sintetico (in-domain)", y=1.01)
plt.tight_layout(); plt.show()

# gap train-test (overfitting)
print("Gap Train-Test (overfitting):")
for nome in sorted(risultati, key=lambda n: risultati[n]["acc_train"] - risultati[n]["acc_test"], reverse=True):
    gap = risultati[nome]["acc_train"] - risultati[nome]["acc_test"]
    print(f"  {nome:<22} gap={gap:+.4f}")

## 5. Esperimento 2 — Domain gap: modelli sintetici testati su foto reali

Testiamo i modelli **già allenati sul sintetico** (Sezione 4) direttamente sulle **foto reali** (train + test insieme, per avere più campioni), senza riaddestrare. Stesso `scaler`.


In [ ]:
# uniamo tutte le reali disponibili (solo valutazione)
X_real_tutte = np.concatenate([X_real_train, X_real_test], axis=0)
y_real_tutte = np.concatenate([y_real_train, y_real_test], axis=0)
X_real_tutte_norm = scaler.transform(X_real_tutte)

print(f"{'Classificatore':<22} {'Acc sintetico':>14} {'Acc reale':>11}")

risultati_gap = {}
for nome, modello in classificatori.items():
    pred = modello.predict(X_real_tutte_norm)
    acc_reale = accuracy_score(y_real_tutte, pred)
    risultati_gap[nome] = {"acc_reale": acc_reale, "pred": pred}
    print(f"{nome:<22} {risultati[nome]['acc_test']:>14.4f} {acc_reale:>11.4f}")

In [ ]:
# confusion matrix sul reale del miglior modello sintetico
migliore = max(risultati, key=lambda n: risultati[n]["acc_test"])
pred = risultati_gap[migliore]["pred"]
print(f"Modello: {migliore} (allenato solo su sintetico)")
print(classification_report(y_real_tutte, pred, target_names=classi, labels=[0, 1], zero_division=0))

cm = confusion_matrix(y_real_tutte, pred, labels=[0, 1])
fig, asse = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Greens", xticklabels=classi, yticklabels=classi, ax=asse)
asse.set_xlabel("Predetto"); asse.set_ylabel("Reale"); asse.set_title(f"Foto reali — {migliore} (solo sintetico)")
plt.tight_layout(); plt.show()

## 6. Esperimento 3 — Mixed training (sintetico + reali di training)

Aggiungiamo le **reali di training** al training set e testiamo sulle **reali di test** (mai viste). Confronto, sullo stesso test reale, tra modello *solo sintetico* e *sintetico + reali*.

> Il "misto" avviene qui in memoria unendo le feature con `np.concatenate`: nessuna cartella mista. Il test resta separato (solo reali di test).


In [ ]:
# training misto in memoria: sintetico train + reali train
X_misto = np.concatenate([X_syn_train, X_real_train], axis=0)
y_misto = np.concatenate([y_syn_train, y_real_train], axis=0)
print("Training misto:", X_misto.shape, " | Test reale:", X_real_test.shape)

# nuovo scaler sul training misto
scaler_misto = StandardScaler()
X_misto_norm      = scaler_misto.fit_transform(X_misto)
X_real_test_norm  = scaler_misto.transform(X_real_test)
# per i modelli solo-sintetico, valutati sullo STESSO test reale
X_real_test_norm_prec = scaler.transform(X_real_test)

print(f"{'Classificatore':<22} {'Reale (solo sint.)':>18} {'Reale (sint.+reali)':>20}")
print("-" * 62)
risultati_misto = {}
for nome, modello_base in classificatori.items():
    # solo sintetico (gia' allenato) sul test reale
    acc_prec = accuracy_score(y_real_test, classificatori[nome].predict(X_real_test_norm_prec))
    # mixed: copia fresca allenata sul misto
    modello = clone(modello_base)
    modello.fit(X_misto_norm, y_misto)
    pred = modello.predict(X_real_test_norm)
    acc_misto = accuracy_score(y_real_test, pred)
    risultati_misto[nome] = {"acc_prec": acc_prec, "acc_misto": acc_misto, "pred": pred}
    print(f"{nome:<22} {acc_prec:>18.4f} {acc_misto:>20.4f}")

In [ ]:
# confusion matrix del miglior modello dopo il mixed training (test reale)
migliore_misto = max(risultati_misto, key=lambda n: risultati_misto[n]["acc_misto"])
pred = risultati_misto[migliore_misto]["pred"]
print(f"Modello: {migliore_misto} (mixed training)")
print(classification_report(y_real_test, pred, target_names=classi, labels=[0, 1], zero_division=0))

cm = confusion_matrix(y_real_test, pred, labels=[0, 1])
fig, asse = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Purples", xticklabels=classi, yticklabels=classi, ax=asse)
asse.set_xlabel("Predetto"); asse.set_ylabel("Reale"); asse.set_title(f"Foto reali test — {migliore_misto} (sint.+reali)")
plt.tight_layout(); plt.show()

# grafico prima/dopo
nomi = list(risultati_misto.keys())
prima = [risultati_misto[n]["acc_prec"]  for n in nomi]
dopo  = [risultati_misto[n]["acc_misto"] for n in nomi]
x = np.arange(len(nomi)); larghezza = 0.35
fig, asse = plt.subplots(figsize=(13, 5))
asse.bar(x - larghezza/2, prima, larghezza, label="Solo sintetico", color="lightgray")
asse.bar(x + larghezza/2, dopo,  larghezza, label="Sintetico + reali", color="mediumpurple")
asse.set_xticks(x); asse.set_xticklabels(nomi, rotation=20, ha="right")
asse.set_ylim(0, 1.05); asse.set_ylabel("Accuracy sul test reale")
asse.set_title("Effetto del mixed training sul domain gap")
asse.legend(); asse.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()

## 7. Conclusioni

1. Sul sintetico imparano al 100% 
Testati sulle stesse immagini sintetiche (dominio su cui sono allenati), quasi tutti fanno 100%. Le classi sono facilmente separabili nello spazio delle feature ResNet.
2. Se addestrato su sintetico e testato su reale, crollano e non riconoscono vest nelle immagini reali
Testati sulle foto reali, collassano: classificano tutto come "senza giubbotto", recall della classe con-giubbotto = 0. Zero giubbotti reali riconosciuti.
Ciò accade perché avevano imparato a riconoscere il giubbotto sintetico (il suo colore/texture specifici del render Unity) e non il concetto generale di giubbotto. I giubbotti reali, con colori e materiali diversi, non attivano quella scorciatoia. Questo è il domain gap: la conoscenza appresa nel dominio sintetico non si trasferisce al reale.
3. Con training misto sanno identificare il giubbotto sui reali
Aggiungendo poche foto reali al training, imparano a riconoscere anche i giubbotti veri: la Logistic Regression passa da recall 0 a 0.93 sul reale, e tutti i modelli salgono a 0.85-0.96 di accuracy. Il gap si chiude.


## 8. Demo (Gradio)

Interfaccia web per provare il classificatore su un'immagine qualsiasi: si trascina una foto e il sistema risponde **with_vest** / **without_vest** con la relativa confidenza.

Usiamo il modello che nel mixed training ha ottenuto la migliore accuracy sul test reale. Pipeline della demo: immagine → ResNet18 (feature 512-dim) → StandardScaler → classificatore.


In [ ]:

migliore_demo = max(risultati_misto, key=lambda n: risultati_misto[n]["acc_misto"])
modello_demo = clone(classificatori[migliore_demo])
modello_demo.fit(X_misto_norm, y_misto)


def probabilita(modello, X):
    if hasattr(modello, "predict_proba"):
        return modello.predict_proba(X)[0]
    punteggio = modello.decision_function(X)[0]
    p_with = 1 / (1 + np.exp(-punteggio))
    return np.array([1 - p_with, p_with])

In [ ]:
import gradio as gr

def classifica_immagine(immagine_pil):
    x = trasformazioni(immagine_pil.convert("RGB")).unsqueeze(0).to(device)
    with torch.no_grad():
        feature = estrattore_feature(x)
        feature = torch.flatten(feature, 1).cpu().numpy()
    feature_norm = scaler_misto.transform(feature)
    p = probabilita(modello_demo, feature_norm)
    return {classi[0]: float(p[0]), classi[1]: float(p[1])}


with gr.Blocks() as demo:
    with gr.Row():
        with gr.Column(scale=1):
            input_img = gr.Image(type="pil", show_label=False, height=400)
        with gr.Column(scale=1):
            output_lbl = gr.Label(num_top_classes=2, show_label=False)

    input_img.change(fn=classifica_immagine, inputs=input_img, outputs=output_lbl)

demo.launch()